# CWD Benchmark — VKG (Ontology) Semantic-Layer Build

This notebook builds a **VKG (virtual knowledge graph / OWL) semantic layer** for
the **CWD (data.world ACME_Insurance)** benchmark database so the deployed
**ontology_query** agent can answer CWD's natural-language questions against a
graph-grounded schema. It mirrors `4_ontology_agent_ac_eval.ipynb` (the insurance
VKG build) and `11_bird_semantic_layer.ipynb`, but CWD is a **single database**
(`cwd_acme`, 13 tables) rather than BIRD's 11 database namespaces — so it builds
**one** VKG layer.

It: seeds a DynamoDB VKG metadata config (mirroring the admin create-layer
wizard), invokes the deployed **ontology generation** agent (`OntologyRuntimeArn`)
which generates an OWL/RDF ontology (N-Quads) from the table schemas, persists
per-table fragments to S3, enriches the Glue Data Catalog, and pushes triples to
Neptune — then blocks on the DynamoDB build status. An LLM-as-Judge
`OntologyQuality` evaluator + `Builtin.GoalSuccessRate` score the generated
ontology **server-side** via AgentCore Batch Evaluations (no local scoring),
exactly as nb4.

### Layer-id convention (IMPORTANT — notebook 14 depends on it)
The single CWD db gets ONE VKG layer, and the DynamoDB `id` is **deterministic**:

> **`layer_id = f"vkg-{namespace}"`** where `namespace = "cwd_acme"` — i.e. the
> layer id is **`vkg-cwd_acme`** (same `vkg-{namespace}` convention as nb11).

`14_cwd_benchmark_eval.ipynb` invokes the ontology_query agent with
`ontologyId = f"vkg-{row.db_id}"` and every CWD row has `db_id == "cwd_acme"`, so
the id resolves to `vkg-cwd_acme` — matching what this notebook builds. Re-running
seeds `status='pending'` over the same id and rebuilds.

### Prerequisites
- **The CWD db is ingested** into S3 Tables / Athena via
  [`scripts/benchmarks/ingest_cwd.py`](../scripts/benchmarks/ingest_cwd.py),
  which writes `data/eval/benchmarks/cwd/manifest.json` mapping the `cwd_acme`
  `db_id` (namespace) → its catalog / database / tables.
- **AgentCore stack** deployed (`{PROJECT_NAME}-agentcore`); `OntologyRuntimeArn`
  populated.
- `bedrock-agentcore>=1.11.0` and `bedrock-agentcore-starter-toolkit==0.3.9`.

> **VKG build is heavy** — the db triggers a two-phase ontology generation
> (per-table N-Quads + holistic FK refinement) **and a Neptune load**. `MAX_DBS`
> (default 1) is kept for parity with nb11 but is a no-op here: CWD has exactly
> one database, so the notebook always builds `cwd_acme`.


In [1]:
# Prereq (uncomment if not already installed):
# !pip install 'bedrock-agentcore>=1.11.0' bedrock-agentcore-starter-toolkit==0.3.9 boto3 pandas python-dotenv --quiet

In [2]:
# Verify the AgentCore Evaluation SDK is available
try:
    import bedrock_agentcore  # noqa: F401
    from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
        BatchEvaluationRunner,  # noqa: F401
    )
    print("✓ bedrock-agentcore installed:",
          getattr(bedrock_agentcore, "__version__", "version unknown"))
    print("✓ BatchEvaluationRunner import OK")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("  Install with: pip install 'bedrock-agentcore>=1.11.0'")
    raise

✓ bedrock-agentcore installed: version unknown
✓ BatchEvaluationRunner import OK


In [3]:
import re

def _redact_account_ids(obj):
    """Recursively mask AWS account IDs in ARN strings to their last 4 digits (********NNNN)."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(\d{8})(\d{4})', r'********\2', obj)
    return obj


def _mask_acct(text):
    """Mask AWS account IDs (12-digit runs) to their last 4 digits — e.g. ********2087 —
    in any string (ARNs, bucket names, plain ids). Never emits the full account number."""
    import re as __re
    if not isinstance(text, str):
        return text
    return __re.sub(r'(\d{8})(\d{4})', r'********\2', text)


## 1. Setup & Credentials

In [4]:
import os
import sys  # noqa: F401  (kept for parity with sibling eval notebooks)
import json
import time
import uuid
import pandas as pd
import boto3
from botocore.config import Config
from datetime import datetime, timezone

from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig, BatchEvaluatorConfig, CloudWatchDataSourceConfig,
)
from bedrock_agentcore.evaluation.runner.dataset_types import (
    Dataset, PredefinedScenario, Turn,
)
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput, AgentInvokerOutput,
)

print("✓ Imports successful")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


✓ Imports successful


In [5]:
# Load environment variables from .env file (sourced from cdk/cdk-outputs-agentcore.json)
from dotenv import load_dotenv, set_key

ENV_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd(), '.env')
load_dotenv(dotenv_path=ENV_FILE, override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')
session = boto3.Session(profile_name=os.environ.get('AWS_PROFILE', 'default'))
region = session.region_name or 'us-east-1'

# ── CloudFormation output resolver ──
# For any env var still set to 'XXX' (no .env or .env missing the key), resolve from
# live CloudFormation stack outputs. Stacks may be deployed with a 'dev' suffix.
def _get_stack_outputs(stack_suffix: str) -> dict:
    """Return {OutputKey: OutputValue} for '{PROJECT_NAME}[-dev]-{suffix}', or {} if absent."""
    cfn = session.client('cloudformation', region_name=region)
    for name in (f'{PROJECT_NAME}-dev-{stack_suffix}', f'{PROJECT_NAME}-{stack_suffix}'):
        try:
            resp = cfn.describe_stacks(StackName=name)
            return {o['OutputKey']: o['OutputValue'] for o in resp['Stacks'][0].get('Outputs', [])}
        except Exception:
            continue
    return {}

_CFN_MAP = {
    'ARTIFACTS_BUCKET':        ('data-lake',  'ArtifactsBucketName'),
    'ATHENA_WORKGROUP':        ('athena',     'AthenaWorkgroupName'),
    'ONTOLOGY_METADATA_TABLE': ('dynamodb',   'MetadataTableName'),
    'KNOWLEDGE_BASE_ID':       ('bedrock-kb', 'OntologyPatternsKbId'),
}
_cfn_cache = {}
_resolved_from_cfn = []
for env_var, (stack, output_key) in _CFN_MAP.items():
    current = os.environ.get(env_var, 'XXX')
    if current != 'XXX':
        continue
    if stack not in _cfn_cache:
        _cfn_cache[stack] = _get_stack_outputs(stack)
    value = _cfn_cache[stack].get(output_key, 'XXX')
    os.environ[env_var] = value
    if value != 'XXX':
        _resolved_from_cfn.append(env_var)
if _resolved_from_cfn:
    for env_var in _resolved_from_cfn:
        set_key(ENV_FILE, env_var, os.environ[env_var])
    print(f"✓ Wrote {len(_resolved_from_cfn)} CFN-resolved variable(s) to .env: {', '.join(_resolved_from_cfn)}")

print("\n✓ Environment variables configured:")
for key in ['ARTIFACTS_BUCKET', 'ATHENA_WORKGROUP', 'ONTOLOGY_METADATA_TABLE', 'KNOWLEDGE_BASE_ID']:
    print(f"  {key} = {_mask_acct(str(os.environ.get(key, '(not set)')))}")


✓ Environment variables configured:
  ARTIFACTS_BUCKET = semantic-layer-dev-artifacts-********4087
  ATHENA_WORKGROUP = semantic-layer-dev-workgroup
  ONTOLOGY_METADATA_TABLE = semantic-layer-dev-metadata
  KNOWLEDGE_BASE_ID = FX2UQDQTQS


In [6]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# Invoke the JWT-inbound runtime with a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    # The deployed function carries the stage infix (e.g. semantic-layer-dev-mcp-tools);
    # try both the bare and the '-dev-' variant, mirroring the auth-stack lookup above.
    cfg = None
    for _fn_name in (f'{PROJECT_NAME}-mcp-tools', f'{PROJECT_NAME}-dev-mcp-tools'):
        try:
            cfg = lam.get_function_configuration(FunctionName=_fn_name)
            break
        except lam.exceptions.ResourceNotFoundException:
            continue
    if cfg is None:
        raise RuntimeError(f'mcp-tools Lambda not found for project {PROJECT_NAME}')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


In [7]:
# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
account_id = identity['Account']
print(f"✓ AWS Credentials Verified:")
print(f"  Profile:   {os.environ.get('AWS_PROFILE', 'default')}")
print(f"  Account:   ...{account_id[-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region:    {region}")

✓ AWS Credentials Verified:
  Profile:   huthmac+demo
  Account:   ...4087
  User/Role: huthmac-Isengard
  Region:    us-east-1


## 2. Resolve the AgentCore Ontology Runtime

In [8]:
ac = _get_stack_outputs('agentcore')
ONTOLOGY_RUNTIME_ARN = ac['OntologyRuntimeArn']
agent_id = ONTOLOGY_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"✓ Runtime ARN: {_mask_acct(ONTOLOGY_RUNTIME_ARN)}")
print(f"✓ Agent ID:    {agent_id}")

✓ Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:********4087:runtime/semantic_layer_dev_ontology-ZUa3LPAcs8
✓ Agent ID:    semantic_layer_dev_ontology-ZUa3LPAcs8


## 3. Discover CWD databases + define per-DB VKG configs

In [9]:
# ── Discover the ingested CWD databases from the benchmark manifest ─────────
# deploy_datasets.py / ingest_cwd.py write a manifest.json mapping each db_id
# (which IS the S3 Tables namespace, e.g. "cwd_acme") to its federated
# catalog, database, and table list. We read it directly rather than re-deriving
# the namespaces live (a live Glue get_tables per namespace is the fallback if a
# manifest is ever absent). Each manifest entry is the exact coordinates the
# ontology agent needs, so no CFN catalog resolution is required here.
import sys
sys.path.insert(0, '..')  # make `scripts` / `agents` importable (mirrors nb2/nb14)
from pathlib import Path
from scripts.benchmarks.ingest_common import Manifest

BENCHMARK = 'cwd'
BENCH_DIR = Path(f'../data/eval/benchmarks/{BENCHMARK}')

# Prefer the ingest_cwd manifest.json (transpile-aware); fall back to the deploy
# driver's deploy_manifest.json. Fail loudly if neither exists — without ingested
# DBs there is nothing to build a layer over.
_manifest_path = None
for _cand in (BENCH_DIR / 'manifest.json', BENCH_DIR / 'deploy_manifest.json'):
    if _cand.exists():
        _manifest_path = _cand
        break
if _manifest_path is None:
    raise FileNotFoundError(
        f"No CWD manifest found under {BENCH_DIR} (looked for manifest.json / "
        f"deploy_manifest.json). Run scripts/benchmarks/deploy_datasets.py "
        f"--benchmark cwd first (see scripts/benchmarks/README.md)."
    )
manifest = Manifest.load(path=_manifest_path)
print(f"✓ Loaded {len(manifest.entries)} ingested CWD db(s) from {_manifest_path.name}")


def s3t(*, catalog: str, namespace: str, table: str) -> dict:
    """Build the S3 Tables (Iceberg) data-source dict the ontology agent expects.

    Parameters:
        catalog: The federated S3 Tables Athena catalog id (from the manifest).
        namespace: The CWD db namespace = Athena database (e.g. cwd_acme).
        table: The table name within the namespace.
    Returns:
        A data-source dict: databaseName=namespace, catalogId=catalog,
        tableId=f"{catalog}::{namespace}.{table}".
    """
    return {
        'databaseName': namespace,
        'tableName': table,
        'catalogId': catalog,
        'dataSource': 'AwsDataCatalog',
        'tableId': f'{catalog}::{namespace}.{table}',
    }


# Generic VKG use-cases text (CWD spans many domains — keep it schema-general so
# the OntologyQuality judge scores STRUCTURE, not insurance-specific modeling).
_VKG_USE_CASES = (
    'Build an OWL ontology (virtual knowledge graph) over the ingested benchmark '
    'database so analysts can ask graph-grounded natural-language questions. Model '
    'one owl:Class per table, an owl:DatatypeProperty per column, and foreign-key '
    'relationships as owl:ObjectProperty with rdfs:domain/rdfs:range, each with a '
    'business-meaningful rdfs:comment.'
)

# Smoke-test slice: MAX_DBS=1 builds a single CWD db (mirrors nb4's MAX_TABLES);
# 0 = build every ingested db (long — a two-phase ontology + Neptune load per db).
MAX_DBS = int(os.environ.get('MAX_DBS', '1'))
_entries = list(manifest.entries)
if MAX_DBS > 0:
    _entries = _entries[:MAX_DBS]
    print(f"  ⚠ MAX_DBS={MAX_DBS} — building a {len(_entries)}-db subset "
          f"(set MAX_DBS=0 to build all {len(manifest.entries)})")

# One test case = one CWD db = one VKG layer = one ontology-agent session.
test_cases = []
for _e in _entries:
    namespace = _e.db_id                 # the manifest keys db_id on the namespace
    layer_id = f'vkg-{namespace}'        # deterministic id nb14 will reuse
    test_cases.append({
        'id': namespace,                 # scenario id = namespace
        'layer_id': layer_id,
        'category': 'cwd_db_ontology',
        'namespace': namespace,
        'catalog': _e.catalog,
        'dynamodb_config': {
            'name': f'cwd-{namespace}-ontology',
            'type': 'VKG',
            'dataSourcesDescription': (
                f'All {len(_e.tables)} Iceberg S3 Tables in the {namespace} '
                f'CWD database namespace.'
            ),
            'useCasesDescription': _VKG_USE_CASES,
            'dataSources': [s3t(catalog=_e.catalog, namespace=namespace, table=t)
                            for t in _e.tables],
        },
    })

if not test_cases:
    raise RuntimeError("No CWD test cases built — manifest has no entries.")

print(f"\n✓ Defined {len(test_cases)} CWD VKG test case(s):")
for tc in test_cases:
    n = len(tc['dynamodb_config']['dataSources'])
    print(f"  {tc['id']} → layer {tc['layer_id']} ({n} table(s))")


✓ Loaded 1 ingested CWD db(s) from manifest.json

✓ Defined 1 CWD VKG test case(s):
  cwd_acme → layer vkg-cwd_acme (13 table(s))


In [10]:
# Helper to seed a semantic-layer-metadata DynamoDB record (replicates the admin UI flow).
METADATA_TABLE = os.environ.get('ONTOLOGY_METADATA_TABLE', 'semantic-layer-metadata')


def upload_enrichment_brief(*, layer_id: str, local_path: str) -> dict:
    """Upload the de-layering enrichment brief to the layer's S3 prefix.

    Mirrors the admin create-layer wizard's document-upload path
    (ontology_service.upload_metadata_file → s3://<artifacts>/ontologies/{id}/uploaded/),
    so the deployed ontology agent lists it under UPLOADED REFERENCE DOCUMENTS and reads
    it while generating each table's OWL/RDF fragment.

    Parameters:
        layer_id: the DynamoDB `id` partition key for this layer (scopes the S3 key).
        local_path: filesystem path to the brief markdown file.

    Returns:
        The uploadedDocuments entry dict: {filename, path (s3 uri), size (bytes)}.
    """
    bucket = os.environ['ARTIFACTS_BUCKET']
    filename = os.path.basename(local_path)
    key = f"ontologies/{layer_id}/uploaded/{filename}"
    session.client('s3', region_name=region).upload_file(local_path, bucket, key)
    return {
        'filename': filename,
        'path': f"s3://{bucket}/{key}",
        'size': os.path.getsize(local_path),
    }


def seed_test_ontology(ontology_id: str, tables: list, *, name: str = 'vkg-eval-test',
                       ontology_type: str = 'VKG', data_sources_description: str = '',
                       use_cases_description: str = '',
                       created_by: str = 'eval@semantic-layer.local',
                       uploaded_documents: list | None = None) -> None:
    """Write a DynamoDB v1 record matching the shape created by the admin UI flow.

    Parameters:
        ontology_id: the `id` partition key the agent will read.
        tables: list of full data-source dicts (from `s3t()`).
        name: human-readable config name; gets a UTC timestamp suffix.
        ontology_type: 'VKG' for the ontology (virtual knowledge graph) agent.
        data_sources_description: optional free-text description of the sources.
        use_cases_description: optional free-text description of the intended use cases.
        created_by: user/email recorded as creator.
        uploaded_documents: optional list of {filename, path, size} reference-document
            dicts (from `upload_enrichment_brief`). Persisted as `uploadedDocuments`,
            which the ontology agent reads on the INITIAL Phase-1 build
            (prompt_builder.build_phase1_table_prompt) — fires on first build.
    """
    ddb_table = session.resource('dynamodb').Table(METADATA_TABLE)
    now = datetime.now(timezone.utc).isoformat()
    ts = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
    name_with_ts = f"{name}-{ts}"
    data_sources = [
        {
            'databaseName': t['databaseName'], 'tableName': t['tableName'],
            'catalogId': t.get('catalogId', 'AWSDataCatalog'),
            'dataSource': t.get('dataSource', 'AwsDataCatalog'),
            'tableId': t.get('tableId', f"{t.get('catalogId', 'AWSDataCatalog')}::{t['databaseName']}.{t['tableName']}"),
        }
        for t in tables
    ]
    item = {
        'id': ontology_id, 'version': 'v1', 'type': ontology_type, 'name': name_with_ts,
        'status': 'pending', 'configuration': {}, 'dataSources': data_sources,
        'createdAt': now, 'updatedAt': now, 'buildStartedAt': now, 'createdBy': created_by,
    }
    if data_sources_description:
        item['dataSourcesDescription'] = data_sources_description
    if use_cases_description:
        item['useCasesDescription'] = use_cases_description
    if uploaded_documents:
        item['uploadedDocuments'] = uploaded_documents
    ddb_table.put_item(Item=item)
    docs_note = f", {len(uploaded_documents)} ref-doc(s)" if uploaded_documents else ""
    print(f"✓ Seeded DynamoDB item: {ontology_id} → name='{name_with_ts}' ({len(data_sources)} table(s){docs_note})")

print("✓ seed_test_ontology + upload_enrichment_brief defined")

✓ seed_test_ontology + upload_enrichment_brief defined


## 4. Create the Custom Ontology-Quality Evaluator — server-side, no Lambda

In [11]:
_cp = session.client('bedrock-agentcore-control', region_name=region)
_SUFFIX = uuid.uuid4().hex[:8]
JUDGE_MODEL_ID = 'global.anthropic.claude-sonnet-5'

# A 0.0–1.0 quality scale (continuous), unlike the binary query-agent judges: ontology
# quality is a matter of degree (coverage / modeling / annotation completeness).
_QUALITY_SCALE = {
    'numerical': [
        {'value': 0.0, 'label': 'poor',
         'definition': 'No owl:Class, almost no properties, or no annotations.'},
        {'value': 0.5, 'label': 'partial',
         'definition': 'Class present but many columns unmodeled, some FKs missed, or shallow comments.'},
        {'value': 1.0, 'label': 'good',
         'definition': 'Full class/property coverage, FK ObjectProperties modeled, business-meaningful rdfs:comments.'},
    ]
}

def _create_llm_judge(*, name: str, level: str, instructions: str) -> str:
    """Create an LLM-as-Judge evaluator and return its evaluatorId.

    Parameters:
        name: human-readable evaluator name (a unique suffix is appended).
        level: 'TRACE' or 'SESSION' — the granularity the service scores at.
        instructions: judge prompt; may reference {context}/{actual_tool_trajectory} etc.

    Returns:
        The created evaluator's evaluatorId.
    """
    resp = _cp.create_evaluator(
        evaluatorName=f"{name}_{_SUFFIX}",
        level=level,
        evaluatorConfig={
            'llmAsAJudge': {
                'instructions': instructions,
                'ratingScale': _QUALITY_SCALE,
                'modelConfig': {
                    'bedrockEvaluatorModelConfig': {
                        'modelId': JUDGE_MODEL_ID,
                        'inferenceConfig': {'maxTokens': 4096},
                    }
                },
            }
        },
    )
    return resp['evaluatorId']


ONTOLOGY_QUALITY_ID = _create_llm_judge(
    name='OntologyQuality',
    level='SESSION',
    instructions=(
        "You are an expert OWL/RDF ontology evaluator scoring a deployed ontology-generation "
        "agent's output for one session covering one or more tables.\n\n"
        "Session context (full conversation, including tool calls and tool results):\n"
        "{context}\n\n"
        "Available tools: {available_tools}\n"
        "Actual tool trajectory: {actual_tool_trajectory}\n\n"
        "In the context, locate:\n"
        "  (a) the OUTPUT of `get_single_table_schema` — the source table columns; and\n"
        "  (b) the ARGUMENTS of `append_nquads` / `save_intermediate_ontology` — the OWL/RDF "
        "N-Quads the agent generated.\n\n"
        "Score the generated ontology on three axes, then return the MEAN as your score:\n"
        "  1. class/property coverage — exactly one owl:Class per table AND an "
        "owl:DatatypeProperty for (nearly) every source column, each with rdfs:domain/rdfs:range.\n"
        "  2. relationship modeling — foreign-key-like columns (e.g. *_id referencing another "
        "table) modeled as owl:ObjectProperty with rdfs:domain/rdfs:range to the related class "
        "(or the table genuinely has no FKs and none are invented).\n"
        "  3. annotation quality — the owl:Class and most properties carry a business-meaningful "
        "rdfs:comment / rdfs:label (not just the column name restated).\n\n"
        "Score 1.0 (good) when all three axes are strong; 0.5 (partial) when coverage is "
        "incomplete, some FKs are missed, or comments are shallow; 0.0 (poor) when there is no "
        "owl:Class, almost no properties, or no annotations. If no N-Quads are present in the "
        "context, score 0.0 and say the generated ontology was not found. Briefly justify the score."
    ),
)

CUSTOM_EVALUATOR_IDS = [ONTOLOGY_QUALITY_ID]
print("✓ Custom server-side evaluator created (LLM-as-Judge, no Lambda):")
print(f"  OntologyQuality (SESSION) : {ONTOLOGY_QUALITY_ID}")

✓ Custom server-side evaluator created (LLM-as-Judge, no Lambda):
  OntologyQuality (SESSION) : OntologyQuality_775d822a-28SWVpCtNL


## 5. Build the Dataset + agent_invoker

In [12]:
# Per-DB VKG-build invoker + dataset (one scenario per CWD db).
#
# The ontology agent is ASYNCHRONOUS (background thread + DynamoDB progress), so
# the invoker must: (1) seed a fresh DynamoDB VKG config under the DETERMINISTIC
# layer id `vkg-<namespace>` (so nb14 can address it), (2) invoke_agent_runtime
# with the framework-managed session_id (so the batch service can correlate
# spans), (3) BLOCK by polling DynamoDB until status is completed/failed, and
# (4) return the build summary. Mirrors nb4's agent_invoker but with NO enrichment
# brief (CWD has no de-layering brief) and a STABLE id (no uuid suffix).
agentcore_client = session.client('bedrock-agentcore', region_name=region)
ddb_resource = session.resource('dynamodb')
ddb_table = ddb_resource.Table(METADATA_TABLE)

SCENARIO_RUNS: dict = {}                       # scenario_id -> build record
BUILT_LAYER_IDS: list = []                     # deterministic vkg-<namespace> ids
_TC_BY_SCENARIO = {tc['id']: tc for tc in test_cases}


def _server_side_seconds(*, item: dict):
    """Compute build latency from the DynamoDB record's own timestamps.

    The agent stamps `buildStartedAt` on the processing transition and
    `completedAt` on completion. Returns None if either is missing/unparseable
    (we never fabricate a value).

    Parameters:
        item: the final DynamoDB item for the ontology build job.
    Returns:
        elapsed seconds as float, or None if timestamps are unavailable.
    """
    started_raw = item.get('buildStartedAt')
    completed_raw = item.get('completedAt')
    if not started_raw or not completed_raw:
        return None
    try:
        return (datetime.fromisoformat(completed_raw)
                - datetime.fromisoformat(started_raw)).total_seconds()
    except (TypeError, ValueError):
        return None



def _glue_column_types(*, glue_client, catalog_id: str, database: str) -> list:
    """Return every column type in a Glue database, as ground truth for the guard.

    The introspection guard needs to know what types the LIVE schema actually
    has before it can judge whether an ontology lost them. Some benchmark
    databases (e.g. bird_toxicology) are legitimately all-string, so the guard
    must not treat all-string ranges as suspicious on their own.

    Parameters:
        glue_client: A boto3 Glue client.
        catalog_id: The Glue catalog id holding the benchmark databases (the
            S3 Tables federated catalog, not the default account catalog).
        database: The Glue database (BIRD/CWD namespace) to introspect.
    Returns:
        list[str]: Lowercased column types across all tables in the database,
            with duplicates retained (callers only test membership/uniqueness).
    Raises:
        RuntimeError: Glue could not be read. The guard cannot verify anything
            without ground truth, so it fails loudly rather than passing.
    """
    types: list = []
    try:
        paginator = glue_client.get_paginator("get_tables")
        for page in paginator.paginate(CatalogId=catalog_id, DatabaseName=database):
            for table in page["TableList"]:
                for column in table["StorageDescriptor"]["Columns"]:
                    types.append(column["Type"].lower())
    except Exception as err:  # noqa: BLE001
        raise RuntimeError(
            f"Cannot read the Glue schema for '{database}' in catalog "
            f"{catalog_id} ({err}), so the schema-introspection guard cannot "
            f"verify the built ontology. Fix the notebook role's Lake Formation / "
            f"Glue access before evaluating."
        ) from err
    if not types:
        raise RuntimeError(
            f"Glue returned no columns for '{database}' in catalog {catalog_id}. "
            f"The database must exist and be readable for the guard to verify "
            f"the ontology."
        )
    return types


def _assert_schema_introspected(
    *,
    layer_id: str,
    s3_client,
    bucket: str,
    glue_client,
    catalog_id: str,
    namespace: str,
) -> None:
    """Fail loudly if a completed ontology was modelled from memory, not from Glue.

    A Lake Formation regression on the ontology build role does NOT fail the build:
    Phase 1 gets AccessDenied per table, and the agent falls back to writing an
    ontology from its own knowledge of the public benchmark schema, then reports
    status='completed'. Such a layer scores like a real one but measures nothing.

    Parameters:
        layer_id: The ontology layer id (e.g. vkg-bird_toxicology).
        s3_client: A boto3 S3 client used to read the built ontology.nq.
        bucket: The artifacts bucket holding ontologies/<layer_id>/ontology.nq.
        glue_client: A boto3 Glue client used to read the REAL source schema,
            which is the ground truth the range check is judged against.
        catalog_id: Glue catalog id of the source database (the account-qualified
            S3 Tables federated catalog, e.g. "<acct>:s3tablescatalog/<bucket>").
        namespace: The source Glue database / benchmark namespace.
    Returns:
        None. Raises instead of returning a status, so callers cannot ignore it.
    Raises:
        RuntimeError: The ontology admits it could not introspect the schema, or
            it flattens to all-xsd:string ranges while the real Glue schema has
            non-string columns, or ontology.nq / the Glue schema is unreadable.
    """
    key = f"ontologies/{layer_id}/ontology.nq"
    try:
        body = s3_client.get_object(Bucket=bucket, Key=key)["Body"].read().decode()
    except Exception as err:  # noqa: BLE001
        raise RuntimeError(
            f"{layer_id} reported completed but s3://{bucket}/{key} is unreadable "
            f"({err}). A completed build must have produced a consolidated ontology."
        ) from err

    # Signal 1: the agent's own admission. Unambiguous — fires on its own.
    if "could not be introspected" in body:
        raise RuntimeError(
            f"{layer_id} was built WITHOUT reading the real schema: ontology.nq "
            f"contains the agent's own 'could not be introspected' admission. This "
            f"means the ontology build role lacks Lake Formation DESCRIBE/SELECT on "
            f"the source database, so the ontology was modelled from the LLM's "
            f"memory of the public benchmark schema. Grant LF on the database AND "
            f"table wildcard to the ontology runtime role (see "
            f"scripts/benchmarks/README.md 'REDEPLOY TRAP'), then rebuild. Refusing "
            f"to evaluate a guessed ontology."
        )

    # Signal 2: ranges that lost type information the REAL schema actually has.
    #
    # An earlier version of this check assumed all-string ranges were themselves
    # proof of hallucination. bird_toxicology disproves that — its real Glue
    # schema is all-string (11 of 11 columns), so all-string ranges are correct
    # there. We therefore compare against ground truth from Glue instead of
    # guessing: only an ontology that flattens types Glue DOES have is suspect.
    ranges = [
        line.split("XMLSchema#")[1].split(">")[0]
        for line in body.splitlines()
        if "rdf-schema#range" in line and "XMLSchema#" in line
    ]
    glue_types = _glue_column_types(
        glue_client=glue_client, catalog_id=catalog_id, database=namespace
    )
    glue_has_non_string = any(t != "string" for t in glue_types)
    if len(ranges) > 1 and set(ranges) == {"string"} and glue_has_non_string:
        raise RuntimeError(
            f"{layer_id}: all {len(ranges)} property ranges are xsd:string, but "
            f"the real Glue schema for '{namespace}' has non-string columns "
            f"({sorted(set(glue_types))}). The ontology therefore did NOT come "
            f"from the live schema — check the ontology role's Lake Formation "
            f"grants (see scripts/benchmarks/README.md 'REDEPLOY TRAP') and "
            f"rebuild. Refusing to evaluate a guessed ontology."
        )
    print(
        f"  ✓ schema introspection verified: {len(ranges)} typed range(s), "
        f"types={sorted(set(ranges))}; glue types={sorted(set(glue_types))}"
    )

def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """Drive the deployed ontology-generation agent for one CWD db and block
    until the async build finishes (so its spans exist for batch discovery).

    BatchEvaluationRunner supplies invoker_input.payload (Turn.input = the
    scenario id = namespace) and invoker_input.session_id (framework session id).
    """
    scenario_id = (invoker_input.payload if isinstance(invoker_input.payload, str)
                   else json.dumps(invoker_input.payload))
    tc = _TC_BY_SCENARIO[scenario_id]
    cfg = tc['dynamodb_config']
    tables = cfg['dataSources']
    num_tables = len(tables)
    layer_id = tc['layer_id']            # DETERMINISTIC — nb14 reuses this exact id

    print(f"\n=== scenario '{scenario_id}' -> layer_id={layer_id} "
          f"({num_tables} tables) ===")
    print(f"  runtimeSessionId = {invoker_input.session_id}")

    seed_test_ontology(
        layer_id, tables, name=cfg['name'], ontology_type=cfg.get('type', 'VKG'),
        data_sources_description=cfg.get('dataSourcesDescription', ''),
        use_cases_description=cfg.get('useCasesDescription', ''),
    )

    _invoke_runtime(
        ONTOLOGY_RUNTIME_ARN,
        invoker_input.session_id,
        json.dumps({'id': layer_id}).encode('utf-8'),
    )

    # Block until the background thread reports completion. Two-phase ontology
    # generation + a Neptune load is slow; budget 150s/table with a 10-min floor.
    # Budget = fixed refinement headroom + 150s per table for the incremental pass.
    # These two costs ADD, so they must not be combined with max(): the build runs
    # `incremental` (one LLM pass per table, scales with table count) and THEN
    # `refinement` (one holistic whole-ontology pass, roughly constant). Measured
    # 2026-08-04 on the 40-table curated layer: incremental ~3420s (~85s/table)
    # plus refinement ~2600s = 6020s total, which overran a `max(3600, 150*40)`
    # = 6000s budget by 20s and aborted the chain on a build that had in fact
    # completed. Additive gives 9600s for 40 tables (under the runner's 14400s
    # cell timeout) and 4050s for a 3-table BIRD db (still clears the refinement
    # stall the floor was introduced for).
    _poll_floor = int(os.environ.get('BUILD_POLL_FLOOR_SECS', '3600'))
    max_wait_secs = _poll_floor + 150 * num_tables
    start = datetime.now()
    final_item, final_status = {}, 'processing'
    print(f"  Polling DynamoDB for completion (max {max_wait_secs}s)...")
    while (datetime.now() - start).total_seconds() < max_wait_secs:
        time.sleep(30)
        final_item = ddb_table.get_item(
            Key={'id': layer_id, 'version': 'v1'}).get('Item', {})
        final_status = final_item.get('status', 'unknown')
        phase = final_item.get('phase', '')
        pct = final_item.get('progressPercent', '')
        suffix = f" — phase={phase} {pct}%" if phase else ""
        print(f"    [{(datetime.now()-start).total_seconds():.0f}s] "
              f"status={final_status}{suffix}")
        if final_status in ('completed', 'failed'):
            break

    summary = final_item.get('summary') or final_item.get('error', '') or f"status={final_status}"
    build_seconds = _server_side_seconds(item=final_item)

    # Verify the ontology was built from the REAL schema before recording success.
    # Done per-build (not once at the end) so a Lake Formation regression aborts on
    # the first database instead of after every build has been paid for.
    if final_status == 'completed':
        # ARTIFACTS_BUCKET is read from the env here rather than a notebook global
        # because the notebook only ever reads it inside seed/upload helpers.
        _assert_schema_introspected(
            layer_id=layer_id,
            s3_client=session.client('s3', region_name=region),
            bucket=os.environ['ARTIFACTS_BUCKET'],
            glue_client=session.client('glue', region_name=region),
            # The manifest stores the catalog WITHOUT the account prefix
            # ("s3tablescatalog/<bucket>"), but Glue's CatalogId needs the
            # account-qualified form, so prepend it unless already present.
            catalog_id=(tc['catalog'] if tc['catalog'].startswith(f'{account_id}:')
                        else f"{account_id}:{tc['catalog']}"),
            namespace=tc['namespace'],
        )

    SCENARIO_RUNS[scenario_id] = {
        'layer_id': layer_id,
        'session_id': invoker_input.session_id,
        'status': final_status,
        'num_tables': num_tables,
        'tables': [t['tableName'] for t in tables],
        'summary': summary,
        'build_seconds': round(build_seconds, 1) if build_seconds is not None else None,
    }
    if final_status == 'completed' and layer_id not in BUILT_LAYER_IDS:
        BUILT_LAYER_IDS.append(layer_id)
    lat = f"{build_seconds:.1f}s" if build_seconds is not None else 'n/a'
    print(f"  {'✓' if final_status == 'completed' else '✗'} {final_status} — {summary}")
    print(f"     latency(server)={lat}")
    return AgentInvokerOutput(agent_output=summary)


# One scenario per CWD db. assertions / expected_trajectory are the ground truth
# the SESSION/span built-in evaluators score against (identical to nb4's).
dataset = Dataset(
    scenarios=[
        PredefinedScenario(
            scenario_id=tc['id'],
            turns=[Turn(input=tc['id'])],     # payload routes the invoker to this tc
            expected_trajectory=[
                'get_single_table_schema',
                'append_nquads',
                'save_intermediate_ontology',
                'update_progress',
            ],
            assertions=[
                "The agent generated an OWL/RDF ontology (owl:Class + owl:DatatypeProperty with rdfs:comment) for every input table.",
                "Foreign-key relationships were modeled as owl:ObjectProperty where applicable.",
                "Per-table N-Quad fragments were persisted via save_intermediate_ontology.",
                "The final DynamoDB status is 'completed'.",
            ],
        )
        for tc in test_cases
    ]
)
print(f"✓ Dataset built: {len(dataset.scenarios)} scenario(s) (one per CWD db)")


✓ Dataset built: 1 scenario(s) (one per CWD db)


## 6. Run the Batch Evaluation (server-side, two-phase + resumable)

In [13]:
import uuid as _uuid
from datetime import datetime, timezone

SERVICE_NAME = f"{agent_id}.DEFAULT"
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"
print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"LOG GROUPS   : {SPANS_LOG_GROUP}, {RUNTIME_LOG_GROUP}")

ALL_EVALUATOR_IDS = [
    'Builtin.GoalSuccessRate',        # SESSION — assertions
    *CUSTOM_EVALUATOR_IDS,            # custom SESSION ontology-quality judge
]

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
    ingestion_delay_seconds=180,  # ontology spans land seconds before the DDB flip
)

# Recovery file: Phase 1 writes the completed session(s) here; Phase 2 reads them.
# Survives a kernel restart, so the long two-phase ontology build never has to be
# re-run just to (re)submit or re-poll the batch. Mirrors notebook 1's Phase 1/2
# split — ported here because ontology generation is even slower (per-table N-Quads
# + holistic FK refinement) and most exposed to a poll/kernel timeout.
SESSIONS_FILE = "../data/eval/results/cwd_vkg_gen_sessions_latest.json"
os.makedirs("../data/eval/results", exist_ok=True)

batch_runner = BatchEvaluationRunner(region=region)

# ── Phase 1: run each scenario's invocation ourselves, then PERSIST ──────────
# Inline (not via run_dataset_evaluation) so the completed build is captured and
# saved to disk BEFORE any batch submission can fail. agent_invoker populates
# SCENARIO_RUNS as a side effect (status / latency / case_id), exactly as the
# results cell expects.
collected_sessions = []   # sessionId / testScenarioId / window / groundTruth
for scenario in dataset.scenarios:
    sid = f"{scenario.scenario_id}-{_uuid.uuid4()}"
    start_time = datetime.now(timezone.utc)
    print(f"\n▶ Phase 1: scenario '{scenario.scenario_id}' (session_id={sid})")
    for turn in scenario.turns:
        agent_invoker(AgentInvokerInput(payload=turn.input, session_id=sid))
    end_time = datetime.now(timezone.utc)
    # Reuse the SDK's own ground-truth transform so assertions / expected_trajectory
    # are encoded identically to run_dataset_evaluation (no drift).
    ground_truth = batch_runner._transform_ground_truth(scenario)
    collected_sessions.append({
        "sessionId": sid,
        "testScenarioId": scenario.scenario_id,
        "startTime": start_time.isoformat(),
        "endTime": end_time.isoformat(),
        "groundTruth": ground_truth,
    })
    print(f"  ✓ session captured: window=[{start_time.isoformat()}, {end_time.isoformat()}]")

with open(SESSIONS_FILE, "w") as f:
    json.dump(
        {
            "agent_id": agent_id,
            "runtime_arn": _mask_acct(ONTOLOGY_RUNTIME_ARN),
            "service_name": SERVICE_NAME,
            "log_groups": [SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
            "evaluator_ids": ALL_EVALUATOR_IDS,
            "sessions": collected_sessions,
            "scenario_runs": SCENARIO_RUNS,
        },
        f, indent=2, default=str,
    )
print(f"\n✓ Phase 1 complete — {len(collected_sessions)} session(s) persisted to {SESSIONS_FILE}")
print("  The ontology build is now saved. Phase 2 (next cell) can submit/re-submit the")
print("  batch cheaply — even after a kernel restart — without re-invoking the agent.")

# Fail loudly on any build that did not reach `completed`. Without this the
# notebook exits 0 having built nothing, BUILT_LAYER_IDS stays empty, and the
# downstream eval notebook scores an empty layer set while still reporting
# success — a silent, expensive-to-spot failure. Raising here also stops us
# paying for a batch evaluation over a broken build.
_incomplete = {
    _sid: _rec.get('status')
    for _sid, _rec in SCENARIO_RUNS.items()
    if _rec.get('status') != 'completed'
}
if _incomplete:
    raise RuntimeError(
        f"{len(_incomplete)} of {len(SCENARIO_RUNS)} layer build(s) did not reach "
        f"status='completed': {_incomplete}. The poll budget may have expired mid-"
        f"refinement (raise BUILD_POLL_FLOOR_SECS) or the agent genuinely failed — "
        f"check the DynamoDB item's `error`/`phase` fields and the runtime logs. "
        f"Refusing to continue: evaluating an incomplete layer produces scores that "
        f"look real but measure nothing."
    )
if not BUILT_LAYER_IDS:
    raise RuntimeError(
        "No layer ids were built, yet every scenario reported completed — this means "
        "the scenario set itself was empty. Check the manifest/table discovery and "
        "MAX_DBS / MAX_TABLES."
    )
print(f"✓ All {len(SCENARIO_RUNS)} build(s) reached status='completed': {BUILT_LAYER_IDS}")

SERVICE_NAME : semantic_layer_dev_ontology-ZUa3LPAcs8.DEFAULT
LOG GROUPS   : aws/spans, /aws/bedrock-agentcore/runtimes/semantic_layer_dev_ontology-ZUa3LPAcs8-DEFAULT



▶ Phase 1: scenario 'cwd_acme' (session_id=cwd_acme-471b8c63-ba9a-42e3-9a19-5d029dde9678)

=== scenario 'cwd_acme' -> layer_id=vkg-cwd_acme (13 tables) ===
  runtimeSessionId = cwd_acme-471b8c63-ba9a-42e3-9a19-5d029dde9678
✓ Seeded DynamoDB item: vkg-cwd_acme → name='cwd-cwd_acme-ontology-20260807-131754' (13 table(s))


  Polling DynamoDB for completion (max 5550s)...


    [30s] status=processing — phase=incremental %


    [60s] status=processing — phase=incremental %


    [90s] status=processing — phase=incremental 7%


... (43 outputs left out)

    [1410s] status=processing — phase=refinement 95%


    [1440s] status=processing — phase=refinement 95%


    [1470s] status=completed — phase=refinement_complete 100%


  ✓ schema introspection verified: 65 typed range(s), types=['long', 'string']; glue types=['bigint', 'string']
  ✓ completed — status=completed
     latency(server)=1443.0s
  ✓ session captured: window=[2026-08-07T13:17:54.231815+00:00, 2026-08-07T13:42:28.719747+00:00]

✓ Phase 1 complete — 1 session(s) persisted to ../data/eval/results/cwd_vkg_gen_sessions_latest.json
  The ontology build is now saved. Phase 2 (next cell) can submit/re-submit the
  batch cheaply — even after a kernel restart — without re-invoking the agent.
✓ All 1 build(s) reached status='completed': ['vkg-cwd_acme']


In [14]:
# ── Phase 2 — Submit + poll the batch (idempotent, re-runnable) ──────────────
# Submits ONE StartBatchEvaluation job over the session(s) captured in Phase 1 and
# polls to completion. SAFE TO RE-RUN ON ITS OWN — it reloads the sessions from
# SESSIONS_FILE if they aren't in memory (e.g. after a kernel restart), so a
# Ctrl-C, poll timeout, or FAILED status never forces a re-invocation of the long
# ontology build. The submit + poll + result-assembly mirror
# run_dataset_evaluation's second half exactly (same pre_evaluation_run_hook span
# wait, same start_batch_evaluation args, same _poll_for_results loop, same
# BatchEvaluationResult shape) — but read sessions from disk, not a live invoke.
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationResult,
    BatchEvaluationSummary,
    CloudWatchOutputDataConfig,
)

POLLING_TIMEOUT_SECONDS = 5400
POLLING_INTERVAL_SECONDS = 30

# Re-runnable: prefer in-memory sessions from Phase 1, else reload from disk.
if "collected_sessions" not in dir() or not collected_sessions:
    with open(SESSIONS_FILE) as f:
        _saved = json.load(f)
    collected_sessions = _saved["sessions"]
    agent_id = _saved["agent_id"]
    ONTOLOGY_RUNTIME_ARN = _saved["runtime_arn"]
    SERVICE_NAME = _saved["service_name"]
    SPANS_LOG_GROUP, RUNTIME_LOG_GROUP = _saved["log_groups"]
    ALL_EVALUATOR_IDS = _saved["evaluator_ids"]
    SCENARIO_RUNS = _saved.get("scenario_runs", {})
    print(f"✓ Reloaded {len(collected_sessions)} session(s) from {SESSIONS_FILE} (Phase 1 not in memory)")

if "batch_runner" not in dir():
    batch_runner = BatchEvaluationRunner(region=region)
if "batch_data_source" not in dir():
    batch_data_source = CloudWatchDataSourceConfig(
        service_names=[SERVICE_NAME],
        log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
        ingestion_delay_seconds=180,
    )

# Parse the persisted session windows back into datetimes.
_session_ids = [s["sessionId"] for s in collected_sessions]
_starts = [datetime.fromisoformat(s["startTime"]) for s in collected_sessions]
_ends = [datetime.fromisoformat(s["endTime"]) for s in collected_sessions]
_session_metadata = [
    {
        "sessionId": s["sessionId"],
        "testScenarioId": s["testScenarioId"],
        **({"groundTruth": {"inline": s["groundTruth"]}} if s.get("groundTruth") else {}),
    }
    for s in collected_sessions
]

batch_name = f"cwd_vkg_gen_batch_{uuid.uuid4().hex[:8]}"
print(f"Batch name : {batch_name}")
print(f"Evaluators : {ALL_EVALUATOR_IDS}")
print(f"Sessions   : {len(_session_ids)}")
print("Waiting for span ingestion, then submitting StartBatchEvaluation...\n")

# 1) Wait for CloudWatch span ingestion (same hook run_dataset_evaluation calls).
batch_data_source.pre_evaluation_run_hook()

# 2) Submit the batch against the EXISTING sessions (no agent invocation here).
_start_response = batch_runner.data_plane_client.start_batch_evaluation(
    batchEvaluationName=batch_name,
    evaluators=[{"evaluatorId": eid} for eid in ALL_EVALUATOR_IDS],
    dataSourceConfig=batch_data_source.to_data_source_config(
        _session_ids, min(_starts), max(_ends),
    ),
    evaluationMetadata={"sessionMetadata": _session_metadata},
    description="Server-side built-in + custom ontology-quality eval of the ontology generation agent.",
)
_batch_id = _start_response["batchEvaluationId"]
_batch_arn = _start_response["batchEvaluationArn"]
print(f"✓ Started batch: {_batch_id}")

# 3) Poll to a terminal state via the SDK's own poll loop.
_response = batch_runner._poll_for_results(
    _batch_id, POLLING_TIMEOUT_SECONDS, POLLING_INTERVAL_SECONDS,
)

# 4) Assemble a BatchEvaluationResult identically to run_dataset_evaluation — a
#    FAILED terminal status still binds batch_result with whatever results exist.
_eval_results = (
    BatchEvaluationSummary.model_validate(_response["evaluationResults"])
    if "evaluationResults" in _response else None
)
_output_data_config = None
if "outputConfig" in _response:
    _odc = _response["outputConfig"].get("cloudWatchConfig")
    if _odc:
        _output_data_config = CloudWatchOutputDataConfig(
            log_group_name=_odc["logGroupName"],
            log_stream_name=_odc["logStreamName"],
        )

batch_result = BatchEvaluationResult(
    batch_evaluation_id=_batch_id,
    batch_evaluation_arn=_batch_arn,
    batch_evaluation_name=_response["batchEvaluationName"],
    status=_response["status"],
    created_at=_response["createdAt"],
    description=_response.get("description"),
    evaluation_results=_eval_results,
    error_details=_response.get("errorDetails"),
    output_data_config=_output_data_config,
)
print(f"\n✓ Batch status: {batch_result.status}")
if batch_result.error_details:
    print(f"⚠ Error details: {batch_result.error_details}")

Batch name : cwd_vkg_gen_batch_cdd1e104
Evaluators : ['Builtin.GoalSuccessRate', 'OntologyQuality_775d822a-28SWVpCtNL']
Sessions   : 1
Waiting for span ingestion, then submitting StartBatchEvaluation...



✓ Started batch: cwd_vkg_gen_batch_cdd1e104-aad06dfef6



✓ Batch status: COMPLETED


## 7. Results — aggregate scores, per-DB detail

In [15]:
os.makedirs('../data/eval/results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f"Batch ID : {batch_result.batch_evaluation_id}")
print(f"ARN      : {_mask_acct(batch_result.batch_evaluation_arn)}")
print(f"Status   : {batch_result.status}")

# ── 1. Aggregate per-evaluator scores ──
agg_rows = []
ev = batch_result.evaluation_results
if ev is not None:
    print(f"\nSessions: completed={ev.number_of_sessions_completed} "
          f"failed={ev.number_of_sessions_failed} total={ev.total_number_of_sessions}")
    for es in (ev.evaluator_summaries or []):
        stats = es.statistics
        avg = (f"{stats.average_score:.3f}"
               if stats and stats.average_score is not None else None)
        agg_rows.append({
            'Evaluator': es.evaluator_id or 'unknown',
            'AvgScore': avg,
            'Evaluated': es.total_evaluated or 0,
            'Failed': es.total_failed or 0,
        })
else:
    print("\n⚠ No aggregate evaluation_results — check job status / spans.")
df_agg = pd.DataFrame(agg_rows)

_EVAL_NAME = {ONTOLOGY_QUALITY_ID: 'OntologyQuality'}

# Map framework session id -> scenario for the per-scenario join.
_SCENARIO_BY_SESSION = {r['session_id']: sid for sid, r in SCENARIO_RUNS.items()}

# ── 2. Per-scenario evaluator events (retry: the output stream lags COMPLETED) ──
events = []
for _attempt in range(6):
    try:
        events = batch_runner.fetch_evaluation_events(batch_result)
        if events:
            break
    except (LookupError, ValueError) as exc:
        print(f"  per-scenario events not ready yet ({exc}); retrying in 20s...")
    time.sleep(20)

def _event_field(e: dict, key: str):
    """Read an event field from the top level or the nested 'attributes' map."""
    if key in e:
        return e[key]
    return (e.get('attributes', {}) or {}).get(key)

event_rows = []
for e in events:
    sess = _event_field(e, 'session.id') or _event_field(e, 'gen_ai.session.id')
    name = _event_field(e, 'gen_ai.evaluation.name')
    event_rows.append({
        'scenario_id': _SCENARIO_BY_SESSION.get(sess, sess),
        'evaluator': _EVAL_NAME.get(name, name),
        'score': _event_field(e, 'gen_ai.evaluation.score.value'),
        'label': _event_field(e, 'gen_ai.evaluation.score.label'),
        'explanation': (str(_event_field(e, 'gen_ai.evaluation.explanation') or '')),
    })
df_events = pd.DataFrame(event_rows)
print(f"\nPer-scenario evaluator events: {len(df_events)}")

# ── 3. Per-scenario build metadata ──
df_runs = pd.DataFrame(list(SCENARIO_RUNS.values()))

_lat = [r['build_seconds'] for r in SCENARIO_RUNS.values() if r.get('build_seconds') is not None]
perf = {
    'scenarios': len(SCENARIO_RUNS),
    'avg_build_seconds': round(sum(_lat) / len(_lat), 1) if _lat else None,
    'total_build_seconds': round(sum(_lat), 1) if _lat else None,
}
print("\n── Build latency (the agent doing the work) ──")
print(f"  Avg server-side latency : {perf['avg_build_seconds']}s  (n={len(_lat)})")

# ── Persist everything ──
combined_file = f"../data/eval/results/cwd_vkg_gen_batch_eval_{timestamp}.json"
combined = {
    'agent_id': agent_id,
    'runtime_arn': _mask_acct(ONTOLOGY_RUNTIME_ARN),
    'batch_evaluation_id': batch_result.batch_evaluation_id,
    'batch_evaluation_arn': batch_result.batch_evaluation_arn,
    'status': str(batch_result.status),
    'evaluator_ids': ALL_EVALUATOR_IDS,
    'custom_evaluators': {'OntologyQuality': ONTOLOGY_QUALITY_ID},
    'aggregate_summaries': agg_rows,
    'per_scenario_events': event_rows,
    'scenario_runs': SCENARIO_RUNS,
    'performance': perf,
}
combined = _redact_account_ids(combined)
with open(combined_file, 'w') as f:
    json.dump(combined, f, indent=2, default=str)
print(f"\n✓ Wrote {combined_file}")

print("\n=== Aggregate per-evaluator scores ===")
display(df_agg)
print("=== Per-scenario evaluator scores ===")
display(df_events)
print("=== Per-scenario build metadata ===")
display(df_runs)

Batch ID : cwd_vkg_gen_batch_cdd1e104-aad06dfef6
ARN      : arn:aws:bedrock-agentcore:us-east-1:********4087:batch-evaluate/cwd_vkg_gen_batch_cdd1e104-aad06dfef6
Status   : COMPLETED

Sessions: completed=1 failed=0 total=1



Per-scenario evaluator events: 2

── Build latency (the agent doing the work) ──
  Avg server-side latency : 1443.0s  (n=1)

✓ Wrote ../data/eval/results/cwd_vkg_gen_batch_eval_20260807_134659.json

=== Aggregate per-evaluator scores ===


,Evaluator,AvgScore,Evaluated,Failed
0,Builtin.GoalSuccessRate,1.000,1,0
1,OntologyQuality_775d822a-28SWVpCtNL,1.000,1,0


=== Per-scenario evaluator scores ===


,scenario_id,evaluator,score,label,explanation
0,cwd_acme,Builtin.GoalSuccessRate,1.0,Yes,"The agent successfully completed all required tasks: (1) OWL/RDF ontology with owl:Class and owl:DatatypeProperty (with rdfs:comment) was generated for all 13 tables including Premium; (2) FK relationships were modeled as owl:ObjectProperty for all applicable tables including Premium's hasPolicyAmount linking to PolicyAmount; (3) Per-table N-Quad fragments were persisted via save_intermediate_ontology for all 13 tables; (4) The specific Phase 2 task for Premium was completed - append_fk_triples returned success:true with fk_triples_added:6, persist_file_to_neptune returned success:true with 26 triples persisted, and update_glue_metadata_from_ontology was called (the AccessDeniedException is noted as expected behavior per the task instructions). The DynamoDB/progress tracking showed 13/13 tables at 100%. The Glue failures are consistent across all tables and are explicitly noted in the task instructions as expected behavior for DynamoDB connector catalogs."
1,cwd_acme,OntologyQuality_775d822a,1.0,good,"The Premium table has a single source column (policy_amount_identifier), and the agent correctly modeled: (1) an owl:Class Premium with a business-meaningful rdfs:comment and label; (2) an owl:DatatypeProperty for the sole column with domain/range and a descriptive comment; (3) the FK relationship (policy_amount_identifier → Policy_Amount) modeled as an owl:ObjectProperty hasPolicyAmount with correct rdfs:domain/rdfs:range and mapsToColumn, appended via append_fk_triples and confirmed success. Neptune persistence succeeded (26 triples). The class also carries rich annotations (businessPurpose, businessConcepts, referenceTables, sampleData, notes) beyond the minimum. The only shortfall across the session is the Glue metadata update failing due to Lake Formation permissions, but the task explicitly treats this as expected/non-blocking and the agent correctly reported it without treating it as a hard failure. All three scoring axes (coverage, relationship modeling, annotation quality) are strong for this table: exactly one class, the one column has a datatype property with domain/range, the FK is modeled as an ObjectProperty, and comments are business-meaningful rather than restated column names."


=== Per-scenario build metadata ===


,layer_id,session_id,status,num_tables,tables,summary,build_seconds
0,vkg-cwd_acme,cwd_acme-471b8c63-ba9a-42e3-9a19-5d029dde9678,completed,13,"[Agreement_Party_Role, Catastrophe, Claim, Claim_Amount, Claim_Coverage, Expense_Payment, Expense_Reserve, Loss_Payment, Loss_Reserve, Policy, Policy_Amount, Policy_Coverage_Detail, Premium]",status=completed,1443.0


## 8. Store Layer IDs for Notebook 14

In [16]:
# Persist the built layer ids so 14_cwd_benchmark_eval can filter its CWD rows
# to the DBs that actually have a completed VKG layer. Written both to the IPython
# %store DB (convenient in-session) and to a small JSON file (survives across
# kernels / notebooks). nb14 also independently scans DynamoDB for completed
# vkg-cwd_* layers, so this file is a convenience, not a hard dependency.
cwd_vkg_layer_ids = list(BUILT_LAYER_IDS)
%store cwd_vkg_layer_ids

layers_file = "../data/eval/results/cwd_vkg_built_layers.json"
os.makedirs("../data/eval/results", exist_ok=True)
with open(layers_file, 'w') as f:
    json.dump({
        'benchmark': 'cwd',
        'ontology_runtime_arn': _mask_acct(ONTOLOGY_RUNTIME_ARN),
        'agent_id': agent_id,
        'layer_id_convention': 'vkg-<namespace> where namespace = cwd_acme (the single CWD db)',
        'built_layer_ids': cwd_vkg_layer_ids,
        'scenario_runs': SCENARIO_RUNS,
    }, f, indent=2, default=str)

print("✓ Stored for notebook 14:")
print(f"  cwd_vkg_layer_ids = {cwd_vkg_layer_ids}")
print(f"  layers file        = {layers_file}")


Stored 'cwd_vkg_layer_ids' (list)
✓ Stored for notebook 14:
  cwd_vkg_layer_ids = ['vkg-cwd_acme']
  layers file        = ../data/eval/results/cwd_vkg_built_layers.json


## Summary

This notebook builds a **VKG (OWL) semantic layer for the single CWD
(ACME_Insurance) database** by driving the deployed ontology-generation agent,
and scores the generated ontology **entirely server-side** via AgentCore Batch
Evaluations — no local scoring.

### Pipeline (mirrors `4_ontology_agent_ac_eval.ipynb` / `11_bird_semantic_layer.ipynb`)
1. **Discover** the ingested CWD db from `data/eval/benchmarks/cwd/manifest.json`
   (namespace `cwd_acme` → catalog / database / tables).
2. **Build**: seed a DynamoDB VKG config under the **deterministic** id
   `vkg-cwd_acme`, invoke the ontology agent, and block on the DynamoDB build
   status. `MAX_DBS` is kept for parity with nb11 but is a no-op (one db).
3. **Evaluators** (server-side, no local scoring): `Builtin.GoalSuccessRate`
   (assertions) + a custom `OntologyQuality` SESSION LLM-as-Judge (class/property
   coverage, FK modeling, `rdfs:comment` quality).
4. **Two-phase, resumable batch** (mirrors nb4): Phase 1 invokes + persists the
   completed session; Phase 2 submits one `StartBatchEvaluation` and polls to
   completion (re-runnable without re-invoking the long build).
5. **Results** → `../data/eval/results/cwd_vkg_gen_batch_eval_*.json`.

### Layer-id convention
`layer_id = "vkg-cwd_acme"` (`vkg-{namespace}`, namespace `cwd_acme`).
`14_cwd_benchmark_eval.ipynb` invokes the ontology_query agent with
`ontologyId = f"vkg-{row.db_id}"` and every CWD row's `db_id` is `cwd_acme`, so
the ids match by construction.
